# 浩劫原生 V-trace 长跑
使用 GPU 运行时（目标 L4）。本地完成正确性与吞吐验证，具体棋力留到这里运行几十小时后评估。Colab L4 尚未实机验证。每 300 秒保存未完成对局到 Google Drive；断线后从最近检查点恢复。

In [ ]:
import sys, subprocess, shutil, os, urllib.request
from pathlib import Path
assert sys.version_info[:2] == (3, 12), '当前包要求 Python 3.12'
REPOSITORY = 'https://github.com/Loremmlel/haojie2.git'
REVISION = input('输入包含本次重写的提交或分支：').strip()
assert REVISION
if not shutil.which('cargo'):
    installer = Path('/content/rustup-init.sh')
    urllib.request.urlretrieve('https://sh.rustup.rs', installer)
    subprocess.run(['sh', str(installer), '-y', '--profile', 'minimal', '--default-toolchain', '1.98.1'], check=True)
    os.environ['PATH'] = str(Path.home() / '.cargo/bin') + os.pathsep + os.environ['PATH']
ROOT = Path('/content/haojie2')
subprocess.run(['git', 'clone', REPOSITORY, str(ROOT)], check=True)
subprocess.run(['git', 'checkout', REVISION], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT / 'training')], check=True)
subprocess.run(['cargo', 'build', '--locked', '--release', '--lib', '--manifest-path', 'native/engine/Cargo.toml', '--target-dir', 'artifacts/native-target'], cwd=ROOT, check=True)


In [ ]:
import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False)
print(torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
from datetime import datetime
RUN = Path('/content/drive/MyDrive/haojie-vtrace') / datetime.now().strftime('run-%Y%m%d-%H%M%S')
RESUME = None  # 恢复时填写可信 checkpoint.pt 的完整路径，并换一个运行目录
command = [sys.executable, '-m', 'haojie_training.online', 'run', '--directory', str(RUN), '--concurrency', '16', '--workers', '2', '--save-seconds', '300']
if RESUME:
    command += ['--resume', str(RESUME)]
process = subprocess.Popen(command, cwd=ROOT)


In [ ]:
def control(action, *args):
    subprocess.run([sys.executable, '-m', 'haojie_training.online', action, '--directory', str(RUN), *args], cwd=ROOT, check=True)
control('status')
# control('resize', '--concurrency', '64')
# control('save')


In [ ]:
control('stop')
process.wait(timeout=120)
print('已保存：', RUN / 'checkpoint.pt')
